### Packages for the Data Generation and Mopdelling of PD (Probability of Default), LGD (Loss Given Default) and EAD (Exposure at Default)

In [1]:
# Data Management and Processing
import pandas as pd
import numpy as np
import scipy
import random

In [2]:
# Machine Learning and Statistics
import sklearn
import statsmodels.api as sm
import tensorflow as tf

2025-03-30 18:30:53.738870: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Data generator

##### Support functions

In [3]:
############# Function to create a profession based on the educational level #########################
def generate_profession(education):
    if education == "high school or lower":
        return random.choices(["LowSkilled", "Unemployed_LowSkilled"], weights=[0.9, 0.1], k=1)[0]
    if education == "ausbildung":
        return random.choices(["MediumSkilled", "Unemployed_MediumSkilled"], weights=[0.9, 0.1], k=1)[0]
    if education in ["bachelor degree", "post graduate degree"]:
        return random.choices(["HighSkilled", "Unemployed_HighSkilled"], weights=[0.9, 0.1], k=1)[0]

In [4]:
############################### Function to generate monthly income and expenditure ##################################                

# Define income parameters for different profession levels and age ranges
income_parameters = {
    ("LowSkilled", "Unemployed_LowSkilled"): {
        (30, 35): {"mean": 1000, "std_dev": 200, "max_income": 2000},
        (36, 40): {"mean": 1200, "std_dev": 200, "max_income": 2400},
        (41, 45): {"mean": 1500, "std_dev": 250, "max_income": 3000},
        (46, 50): {"mean": 1800, "std_dev": 300, "max_income": 3600},
        (51, 55): {"mean": 2000, "std_dev": 350, "max_income": 4000},
        (56, 60): {"mean": 2200, "std_dev": 400, "max_income": 4300},
        (61, 65): {"mean": 2400, "std_dev": 600, "max_income": 4500},
    },
    ("MediumSkilled", "Unemployed_MediumSkilled"): {
        (30, 35): {"mean": 1800, "std_dev": 300, "max_income": 4000},
        (36, 40): {"mean": 2300, "std_dev": 400, "max_income": 5000},
        (41, 45): {"mean": 2600, "std_dev": 500, "max_income": 6000},
        (46, 50): {"mean": 3000, "std_dev": 600, "max_income": 7000},
        (51, 55): {"mean": 3500, "std_dev": 800, "max_income": 8000},
        (56, 60): {"mean": 4000, "std_dev": 800, "max_income": 9000},
        (61, 65): {"mean": 5000, "std_dev": 1000, "max_income": 10000},
    },
    ("HighSkilled", "Unemployed_HighSkilled"): {
        (30, 35): {"mean": 3000, "std_dev": 500, "max_income": 10000},
        (36, 40): {"mean": 4500, "std_dev": 700, "max_income": 15000},
        (41, 45): {"mean": 6000, "std_dev": 1000, "max_income": 20000},
        (46, 50): {"mean": 7000, "std_dev": 1500, "max_income": 25000},
        (51, 55): {"mean": 8000, "std_dev": 2000, "max_income": 30000},
        (56, 60): {"mean": 9000, "std_dev": 3000, "max_income": 40000},
        (61, 65): {"mean": 10000, "std_dev": 4000, "max_income": 50000},
    }
}

# To calculate the monthly income and expenditure
def generate_income_expense(profession_undertake, current_age, num_dependents):
    # Iterate over income parameters for each profession group
    for profession_group, age_ranges in income_parameters.items():
        # Check if profession_undertake is one of the professions in the profession_group tuple
        if profession_undertake in profession_group:
            # Iterate over the age ranges and income parameters
            for age_range, params in age_ranges.items():
                if age_range[0] <= current_age <= age_range[1]:
                    mean_income = params["mean"]
                    std_dev = params["std_dev"]
                    max_income = params["max_income"]
                    unemployement_money = mean_income * 0.5  # 50% of mean income for unemployment

                    # If profession_undertake contains the word "Unemployed" before "_", return the unemployment money
                    if profession_undertake.split("_")[0] == "Unemployed":
                        income = unemployement_money
                        expenditure = generate_expenditure(income, mean_income, num_dependents) # The belong to the population under mean_income
                        return income, expenditure
                    
                    # If profession_undertake does not contain the word "Unemployed", generate income with an specific rule
                    else:
                        calc_income = int(np.random.normal(mean_income, std_dev))
                        income = max(unemployement_money, min(calc_income, max_income))  
                        expenditure = generate_expenditure(income, mean_income, num_dependents)
                        return income, expenditure

# Support function to calculate the expenditure
def generate_expenditure(income, mean_income, num_dependents):
    # Values for low and for high income people (lower possible value, mode, higher possible value) depending on the number of dependents
    low_income_params = [(0.5, 0.7, 1.5), (0.7, 0.8, 1.5), (0.8, 0.9, 1.5), (0.9, 0.9, 1.5), (0.9, 1.0, 1.5)]
    high_income_params = [(0.5, 0.7, 1.5), (0.6, 0.7, 1.5), (0.7, 0.75, 1.5), (0.7, 0.75, 1.5), (0.8, 0.85, 1.5)]
    # The parameters that should be taken depend on whether the income is below or above the mean income
    params = low_income_params if income < mean_income else high_income_params
    # Here we recover the parameters
    left, mode, right = params[min(num_dependents, 4)]  # Ensure index stays within range
    # The expenditure is calculated given a rule of min, mode, max
    expenditure = income * np.random.triangular(left=left, mode=mode, right=right)
    return expenditure



In [5]:
############################### Function to generate the credit to be requested ##################################

In [5]:
############################### Function to generate the Y-variable (default 0, 1) ##################################     
def default_y_calculation(profession, debt_income_ratio):
    # past_credits, dependents, professon, debt_income_ratio
    if profession == ("Unemployed_LowSkilled" or "Unemployed_MediumSkilled") and debt_income_ratio > 0.1:
        return 1
    if profession == ("Unemployed_HighSkilled") and debt_income_ratio > 0.2:
        return 1     


##### Data Generator for the original state of individuals

In [6]:
# Function to generate data accordingly to some requirements
def data_generator(number_of_customers):
    data = []
    for i in range(number_of_customers):
        
        # ---------------- X-Variables -------------------------------#
        ##### Variables not directly dependent on other variables #####
        name = f"name{i}" # names are created according to the index "i"
        age = random.randint(30, 60) # As the maximum attainable age that we want in the game is 65
        education_level = random.choices(["high school or lower", "ausbildung", "bachelor degree", "post graduate degree"],  weights=[0.3, 0.3, 0.3, 0.1], k=1)[0]
        # Number of unpaid past credits
        past_credits = random.choices([0, 1, 2, 3], weights=[0.6, 0.3, 0.08, 0.02], k=1)[0] # This emphasizes 0 and 1 unpaid credits
        # Number of dependents
        dependents = random.choices([0, 1, 2, 3, 4], weights=[0.6, 0.3, 0.06, 0.03, 0.01], k=1)[0] # This emphasizes 0 and 1 unpaid credits
        
        ##### Variables directly dependent on other variables #####
        # Generate profession based on education level
        profession = generate_profession(education_level)
        # Generate monthly income based on profession, age and number of dependents
        monthly_income = generate_income_expense(profession, age, dependents)[0]
        # Generate monthly expenditure dependening on the income, mean income and number of dependents
        monthly_expenditure = generate_income_expense(profession, age, dependents)[1]
        
        ##### Other variables generated from the variables above #####
        savings_debt = monthly_income - monthly_expenditure
        # Debt to income ratio
        if savings_debt < 0:
            #debt_to_income_ratio = f"{abs(savings_debt/monthly_income):.2%}"
            debt_to_income_ratio = abs(savings_debt/monthly_income)
        else:
            #debt_to_income_ratio = f"{0:.2%}"
            debt_to_income_ratio = 0
        
        # ---------------- Credit amount requested and time of the request--------------------------------#
        
        
        # ---------------- Y-Variable --------------------------------#
        
        data.append({
            'name': name,
            'age': age,
            'educational level': education_level,
            'number of not paid past credits': past_credits,
            'dependents': dependents,
            'profession': profession,
            'monthly income': monthly_income,
            'monthly expenditure': monthly_expenditure,
            'savings (debt)': savings_debt,
            'debt-to-income ratio': debt_to_income_ratio
        })

    # Create a pandas DataFrame
    df = pd.DataFrame(data)
    return df

##### Generating one data frame

In [7]:
number_of_customers_1 = 1000
df_1 = data_generator(number_of_customers_1)
df_1

,name,age,educational level,number of not paid past credits,dependents,profession,monthly income,monthly expenditure,savings (debt),debt-to-income ratio
0,name0,32,high school or lower,0,0,LowSkilled,973.0,796.110962,176.889038,0.000000
1,name1,59,high school or lower,0,0,LowSkilled,2525.0,2312.829446,212.170554,0.000000
2,name2,35,ausbildung,0,0,MediumSkilled,1667.0,1410.936117,256.063883,0.000000
3,name3,39,ausbildung,0,1,MediumSkilled,2380.0,2529.300958,-149.300958,0.062731
4,name4,58,bachelor degree,0,0,HighSkilled,15655.0,8090.301024,7564.698976,0.000000
...,...,...,...,...,...,...,...,...,...,...
995,name995,39,bachelor degree,0,0,HighSkilled,4836.0,3493.215669,1342.784331,0.000000
996,name996,52,ausbildung,1,1,MediumSkilled,3653.0,2523.566556,1129.433444,0.000000
997,name997,45,bachelor degree,1,4,HighSkilled,5390.0,5019.337782,370.662218,0.000000
998,name998,38,ausbildung,1,1,Unemployed_MediumSkilled,1150.0,1237.281381,-87.281381,0.075897


##### DF statistics

In [8]:
df_1.describe()

,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio
count,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000
mean,45.194000,0.522000,0.56800,3619.082000,3390.243212,228.838788,0.134172
std,9.110838,0.715561,0.87872,2669.532846,2617.733492,1853.844315,0.277080
min,30.000000,0.000000,0.00000,500.000000,289.205524,-12818.073647,0.000000
25%,37.000000,0.000000,0.00000,1740.750000,1559.395180,-331.003857,0.000000
50%,45.000000,0.000000,0.00000,2778.500000,2584.905299,184.731428,0.000000
75%,53.000000,1.000000,1.00000,4606.250000,4463.644007,860.415079,0.153417
max,60.000000,3.000000,4.00000,15655.000000,17113.073647,9927.854029,2.984418


Monthly income by profession

In [9]:
df_1.groupby('profession')['monthly income'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,359.0,6142.598886,2811.418294,2214.0,3845.5,5616.0,7904.50,15655.0
LowSkilled,237.0,1590.396624,489.557777,547.0,1207.0,1559.0,1947.00,3079.0
MediumSkilled,300.0,2838.883333,985.822572,1181.0,2088.5,2726.5,3450.25,6696.0
Unemployed_HighSkilled,39.0,2974.358974,1111.830626,1500.0,2250.0,3000.0,4000.00,4500.0
Unemployed_LowSkilled,36.0,797.222222,228.017682,500.0,575.0,900.0,1000.00,1100.0
Unemployed_MediumSkilled,29.0,1400.000000,370.328040,900.0,1150.0,1300.0,1750.00,2000.0


Debt-to-income ratio by profession

In [10]:
df_1.groupby('profession')['debt-to-income ratio'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,359.0,0.176275,0.357508,0.0,0.0,0.0,0.183958,2.984418
LowSkilled,237.0,0.107612,0.217390,0.0,0.0,0.0,0.125526,1.330679
MediumSkilled,300.0,0.129614,0.239894,0.0,0.0,0.0,0.159466,1.615764
Unemployed_HighSkilled,39.0,0.081590,0.136989,0.0,0.0,0.0,0.093481,0.455611
Unemployed_LowSkilled,36.0,0.051819,0.084946,0.0,0.0,0.0,0.078240,0.300731
Unemployed_MediumSkilled,29.0,0.050115,0.093951,0.0,0.0,0.0,0.056943,0.309018
